# Minimum Variance Portfolio Optimization — Version 2
## Rolling Backtest and Robust Portfolio Framework

**Author:** Thomas Christian Matenco  
**Course:** Algorithmic Trading, IE University MBDS  
**Version:** 2.1 — Rolling Backtest Extension (corrected)

---

This notebook extends the single-period Version 1 analysis into a full rolling-window backtest framework.

**What Version 2 adds over Version 1:**
- Rolling-window backtesting across five out-of-sample years (2019–2023)
- Four portfolio construction strategies compared side by side
- Rebalancing logic (annual, quarterly, monthly)
- Transaction cost modelling including year-boundary turnover
- Risk contribution analysis
- Extended performance metrics (Sortino, Calmar, VaR, CVaR, Hit Rate)
- Strategy ranking table
- Full visualisation suite

**Assets:** ACWI (global equities) | AGG (US aggregate bonds) | GLD (gold) | BSV (short-term bonds)  
**Benchmark:** Equal-weight Permanent Portfolio (25% each)  
**Training window:** 4 years rolling, expanding within year on rebalance dates  
**Test periods:** 2019, 2020, 2021, 2022, 2023  
**Risk-free rate:** 0% (consistent with Version 1)

---

**Training window note:** At the start of each test year, the covariance matrix is estimated on the prior 4 calendar years. For intra-year rebalancing tests, estimates are updated on an expanding window using all data available up to the prior rebalance date. No future data is used at any point.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import warnings

from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
pd.set_option("display.max_columns", 20)

TRADING_DAYS = 252
RISK_FREE    = 0.00
TICKERS      = ["ACWI", "AGG", "GLD", "BSV"]
MAX_WEIGHT   = 0.40
TRAIN_YEARS  = 4
TEST_YEARS   = [2019, 2020, 2021, 2022, 2023]
TC_BPS       = 5   # basis points per unit of one-way turnover

# Strategy colours
COLORS = {
    "min_variance":  "#111111",
    "risk_parity":   "#555555",
    "max_sharpe":    "#999999",
    "equal_weight":  "#CCCCCC",
}
# Asset colours (separate from strategy colours — fix #5)
ASSET_COLORS = {
    "ACWI": "#111111",
    "AGG":  "#555555",
    "GLD":  "#999999",
    "BSV":  "#CCCCCC",
}
LABELS = {
    "min_variance": "Min Variance (40% cap)",
    "risk_parity":  "Risk Parity (unconstrained)",
    "max_sharpe":   "Max Sharpe (unconstrained)",
    "equal_weight": "Equal Weight (Benchmark)",
}
LS = {
    "min_variance": "-",
    "risk_parity":  "--",
    "max_sharpe":   "-.",
    "equal_weight": ":",
}

## 2. Data Download

Data runs from 2015 to support 4-year rolling training windows starting with the first test year (2019).

In [ ]:
prices = yf.download(
    TICKERS,
    start="2015-01-01",
    end="2024-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

prices = prices.reindex(columns=TICKERS).dropna()
assert list(prices.columns) == TICKERS

print(f"Downloaded: {prices.shape[0]} trading days")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
print()
for yr in TEST_YEARS:
    n_train = len(prices.loc[f"{yr-TRAIN_YEARS}-01-01":f"{yr-1}-12-28"])
    n_test  = len(prices.loc[f"{yr}-01-01":f"{yr}-12-31"])
    print(f"  {yr}  train observations: {n_train:4d}  |  test observations: {n_test:3d}")

In [ ]:
returns = prices.pct_change().dropna()

# Normalized price chart — uses ASSET_COLORS (fix #5)
norm = prices / prices.iloc[0] * 100
fig, ax = plt.subplots(figsize=(13, 5))
for t in TICKERS:
    ax.plot(norm.index, norm[t], label=t, linewidth=1.8, color=ASSET_COLORS[t])
for yr in TEST_YEARS:
    ax.axvline(pd.Timestamp(f"{yr}-01-01"), color="grey", linewidth=0.6, linestyle="--", alpha=0.4)
ax.set_title("Normalized Price Performance (Base = 100, Jan 2015)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Index Value")
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

## 3. Core Functions

**Constraint note (fix #3):** The 40% per-asset cap is applied only to the Minimum Variance portfolio, matching the Version 1 submitted specification. Risk Parity and Maximum Sharpe are shown as unconstrained long-only strategies to demonstrate the effect of the optimizer without an imposed limit. Where constrained variants are relevant, they can be activated by passing `max_weight` into those optimizers.

In [ ]:
def ledoit_wolf_cov(returns_df):
    """Annualised Ledoit-Wolf covariance matrix."""
    lw = LedoitWolf().fit(returns_df.values)
    return lw.covariance_ * TRADING_DAYS, lw.shrinkage_


def minimize_variance(cov_matrix, max_weight=MAX_WEIGHT):
    """Long-only minimum variance with per-asset cap (default 40%)."""
    n = cov_matrix.shape[0]
    result = minimize(
        lambda w: np.sqrt(w.T @ cov_matrix @ w),
        x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, max_weight)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success: raise ValueError(f"MinVar failed: {result.message}")
    return result.x


def maximize_sharpe(exp_returns_ann, cov_matrix, rf=RISK_FREE):
    """Long-only maximum Sharpe, unconstrained (no per-asset cap)."""
    n = len(exp_returns_ann)
    def neg_sharpe(w):
        vol = np.sqrt(w.T @ cov_matrix @ w)
        return -(w @ exp_returns_ann - rf) / vol if vol > 1e-10 else 1e10
    result = minimize(
        neg_sharpe, x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success: raise ValueError(f"MaxSharpe failed: {result.message}")
    return result.x


def risk_contributions(weights, cov_matrix):
    """Percentage contribution of each asset to total portfolio risk."""
    port_vol = np.sqrt(weights.T @ cov_matrix @ weights)
    if port_vol < 1e-10: return np.zeros_like(weights)
    marginal = cov_matrix @ weights / port_vol
    rc = weights * marginal
    return rc / rc.sum()


def risk_parity_objective(weights, cov_matrix):
    rc = risk_contributions(weights, cov_matrix)
    target = np.repeat(1/len(weights), len(weights))
    return np.sum((rc - target) ** 2)


def optimize_risk_parity(cov_matrix):
    """Long-only risk parity, unconstrained (no per-asset cap)."""
    n = cov_matrix.shape[0]
    result = minimize(
        lambda w: risk_parity_objective(w, cov_matrix),
        x0=np.repeat(1/n, n), method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1}
    )
    if not result.success: raise ValueError(f"RiskParity failed: {result.message}")
    return result.x


def performance_metrics(daily_returns, rf=RISK_FREE):
    """Full performance metric suite."""
    ann_ret = daily_returns.mean() * TRADING_DAYS
    ann_vol = daily_returns.std()  * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - rf) / ann_vol if ann_vol > 0 else np.nan
    downside = daily_returns[daily_returns < 0]
    dv = downside.std() * np.sqrt(TRADING_DAYS) if len(downside) > 1 else np.nan
    sortino = (ann_ret - rf) / dv if dv and dv > 0 else np.nan
    cum  = (1 + daily_returns).cumprod()
    mdd  = ((cum - cum.cummax()) / cum.cummax()).min()
    calmar = ann_ret / abs(mdd) if mdd != 0 else np.nan
    var95  = daily_returns.quantile(0.05)
    cvar95 = daily_returns[daily_returns <= var95].mean()
    return {
        "Ann. Return":    ann_ret,   "Ann. Volatility": ann_vol,
        "Sharpe":         sharpe,    "Sortino":         sortino,
        "Calmar":         calmar,    "Max Drawdown":    mdd,
        "Cum. Return":    cum.iloc[-1] - 1,
        "VaR 95%":        var95,     "CVaR 95%":        cvar95,
        "Hit Rate":       (daily_returns > 0).mean(),
    }


def calculate_turnover(old_weights, new_weights):
    return float(np.sum(np.abs(new_weights - old_weights)))

print("All functions defined.")

## 4. Rolling Backtest Engine

**Transaction cost treatment (fix #2):** Transaction costs are applied at two points:
1. **Year-boundary turnover** — when moving from the prior year's final weights to the new year's initial weights (charged on the first test day).
2. **Intra-year rebalancing** — when rebalancing within the test year.

This makes the cost modelling more realistic than applying costs only to intra-year trades.

**Training window logic (fix #1):**
- At the start of each test year: 4-year rolling window ending December 28 of prior year.
- On intra-year rebalance dates: expanding window from the original training start to the prior rebalance date.

In [ ]:
def rolling_backtest(
    returns,
    tickers,
    train_years=TRAIN_YEARS,
    test_years=TEST_YEARS,
    methods=None,
    max_weight=MAX_WEIGHT,
    rebalance="annual",
    transaction_cost_bps=TC_BPS,
    rf=RISK_FREE,
    verbose=True,
):
    """
    Rolling-window portfolio backtest across multiple out-of-sample years.

    Parameters
    ----------
    returns              : pd.DataFrame  Daily asset returns
    tickers              : list          Asset tickers
    train_years          : int           Years of training data at year-start
    test_years           : list          Out-of-sample years to evaluate
    methods              : list          Strategies to run
    max_weight           : float         Per-asset cap for MinVar only
    rebalance            : str           annual | quarterly | monthly | none
    transaction_cost_bps : float         Bps per unit of one-way turnover
    rf                   : float         Risk-free rate
    verbose              : bool          Print progress

    Returns
    -------
    port_return_series : dict  {method: pd.Series}
    wt_hist            : dict  {method: pd.DataFrame of year-end weights}
    met_hist           : dict  {method: pd.DataFrame of annual metrics}
    turnover_df        : pd.DataFrame  Annual turnover by method
    """
    if methods is None:
        methods = ["min_variance", "risk_parity", "max_sharpe", "equal_weight"]

    all_daily  = {m: [] for m in methods}
    wt_records = {m: [] for m in methods}
    mt_records = {m: [] for m in methods}
    to_records = []

    # Track prior-year final weights to charge year-boundary costs (fix #2)
    prev_weights = {m: None for m in methods}

    def get_weights(m, cov, exp_r):
        if   m == "min_variance":  return minimize_variance(cov, max_weight)
        elif m == "risk_parity":   return optimize_risk_parity(cov)
        elif m == "max_sharpe":    return maximize_sharpe(exp_r, cov, rf)
        elif m == "equal_weight":  return np.repeat(1/len(tickers), len(tickers))
        else: raise ValueError(f"Unknown method: {m}")

    for test_year in test_years:
        train_start = f"{test_year - train_years}-01-01"
        train_end   = f"{test_year - 1}-12-28"
        test_start  = f"{test_year}-01-01"
        test_end    = f"{test_year}-12-31"

        ret_train = returns.loc[train_start:train_end, tickers]
        ret_test  = returns.loc[test_start:test_end, tickers]

        if len(ret_train) < 100 or len(ret_test) < 20:
            if verbose: print(f"  {test_year}: insufficient data, skipping")
            continue

        if verbose: print(f"  {test_year}: train={len(ret_train)}, test={len(ret_test)}")

        cov_ann, shrinkage = ledoit_wolf_cov(ret_train)
        exp_ret_ann = ret_train.mean().values * TRADING_DAYS

        init_weights = {m: get_weights(m, cov_ann, exp_ret_ann) for m in methods}

        # Intra-year rebalance schedule
        if rebalance == "quarterly":
            rb_set = set(ret_test.resample("QS").first().index.normalize())
        elif rebalance == "monthly":
            rb_set = set(ret_test.resample("MS").first().index.normalize())
        else:
            rb_set = set()   # annual/none: no intra-year rebalances

        to_row = {"year": test_year}

        for m in methods:
            current_w = init_weights[m].copy()
            day_rets  = []
            total_to  = 0.0

            for i, date in enumerate(ret_test.index):
                tc_cost = 0.0

                if i == 0:
                    # Year-boundary cost: charge turnover from prior year's final weights (fix #2)
                    if prev_weights[m] is not None:
                        to_yrb = calculate_turnover(prev_weights[m], current_w)
                        tc_cost = to_yrb * transaction_cost_bps / 10000
                        total_to += to_yrb

                elif date.normalize() in rb_set:
                    # Intra-year rebalance: expanding window up to previous trading day
                    prior_date = ret_test.index[i-1].strftime("%Y-%m-%d")
                    ret_up = returns.loc[train_start:prior_date, tickers]
                    if len(ret_up) > 100:
                        cov_up, _ = ledoit_wolf_cov(ret_up)
                        exp_up    = ret_up.mean().values * TRADING_DAYS
                        new_w     = get_weights(m, cov_up, exp_up)
                        to_intra  = calculate_turnover(current_w, new_w)
                        tc_cost   = to_intra * transaction_cost_bps / 10000
                        total_to += to_intra
                        current_w = new_w

                dr = float(ret_test.loc[date].values @ current_w) - tc_cost
                day_rets.append(dr)

            # Store prior year's final weights for next year's boundary cost
            prev_weights[m] = current_w.copy()

            series = pd.Series(day_rets, index=ret_test.index, name=m)
            all_daily[m].append(series)
            wt_records[m].append({"year": test_year, **dict(zip(tickers, current_w))})
            mets = performance_metrics(series, rf=rf)
            mets["year"] = test_year
            mets["LW shrinkage"] = shrinkage
            mt_records[m].append(mets)
            to_row[m] = total_to

        to_records.append(to_row)

    port_return_series = {m: pd.concat(all_daily[m])         for m in methods if all_daily[m]}
    wt_hist            = {m: pd.DataFrame(wt_records[m]).set_index("year")  for m in methods if wt_records[m]}
    met_hist           = {m: pd.DataFrame(mt_records[m]).set_index("year")  for m in methods if mt_records[m]}
    turnover_df        = pd.DataFrame(to_records).set_index("year") if to_records else pd.DataFrame()

    return port_return_series, wt_hist, met_hist, turnover_df

print("Rolling backtest engine ready.")

## 5. Run the Rolling Backtest

In [ ]:
METHODS = ["min_variance", "risk_parity", "max_sharpe", "equal_weight"]

print("Running rolling backtest...")
print(f"  Methods:     {METHODS}")
print(f"  Test years:  {TEST_YEARS}")
print(f"  Train years: {TRAIN_YEARS}")
print(f"  Rebalance:   annual")
print(f"  TC:          {TC_BPS} bps per unit turnover")
print()

port_rets, wt_hist, met_hist, turnover_df = rolling_backtest(
    returns=returns, tickers=TICKERS,
    train_years=TRAIN_YEARS, test_years=TEST_YEARS,
    methods=METHODS, max_weight=MAX_WEIGHT,
    rebalance="annual", transaction_cost_bps=TC_BPS,
    verbose=True,
)
print("\nBacktest complete.")

## 6. Annual Performance Tables

In [ ]:
# Annual Sharpe table
sharpe_tbl = pd.DataFrame({LABELS[m]: met_hist[m]["Sharpe"] for m in METHODS if m in met_hist})
print("Annual Sharpe Ratios:")
display(sharpe_tbl.round(4))

In [ ]:
# Annual return table
ret_tbl = pd.DataFrame({LABELS[m]: met_hist[m]["Ann. Return"].map("{:.2%}".format) for m in METHODS if m in met_hist})
print("Annual Returns:")
display(ret_tbl)

In [ ]:
# Annual max drawdown table
dd_tbl = pd.DataFrame({LABELS[m]: met_hist[m]["Max Drawdown"].map("{:.2%}".format) for m in METHODS if m in met_hist})
print("Annual Max Drawdown:")
display(dd_tbl)

In [ ]:
# Turnover table
if not turnover_df.empty:
    to_disp = turnover_df.copy()
    for m in METHODS:
        if m in to_disp.columns:
            to_disp[m] = to_disp[m].map("{:.2%}".format)
    to_disp.columns = [LABELS[m] for m in METHODS if m in to_disp.columns]
    print("Annual Turnover (one-way, includes year-boundary cost):")
    display(to_disp)

In [ ]:
# Full-period aggregate metrics (keep raw version for ranking)
print("Full-Period Aggregate Performance (2019-2023):")
agg_rows = {}
for m in METHODS:
    if m in port_rets:
        agg_rows[LABELS[m]] = performance_metrics(port_rets[m])

agg_raw = pd.DataFrame(agg_rows).T   # raw floats for ranking (fix #8)
agg_df  = agg_raw.copy()

fmt_pct = ["Ann. Return","Ann. Volatility","Max Drawdown","Cum. Return","VaR 95%","CVaR 95%","Hit Rate"]
fmt_num = ["Sharpe","Sortino","Calmar"]
for col in fmt_pct:
    if col in agg_df.columns: agg_df[col] = agg_df[col].map("{:.2%}".format)
for col in fmt_num:
    if col in agg_df.columns: agg_df[col] = agg_df[col].map("{:.4f}".format)

display(agg_df)

## 7. Strategy Ranking (Full Period)

Ranking each strategy on the four most important metrics. Rank 1 = best.

In [ ]:
rank_df = pd.DataFrame({
    "Sharpe Rank":     agg_raw["Sharpe"].rank(ascending=False),
    "Return Rank":     agg_raw["Ann. Return"].rank(ascending=False),
    "Volatility Rank": agg_raw["Ann. Volatility"].rank(ascending=True),   # lower = better
    "Drawdown Rank":   agg_raw["Max Drawdown"].rank(ascending=False),      # less negative = better
    "Sortino Rank":    agg_raw["Sortino"].rank(ascending=False),
})
rank_df["Average Rank"] = rank_df.mean(axis=1).round(2)
rank_df = rank_df.sort_values("Average Rank")
display(rank_df)

## 8. Portfolio Weights Through Time

In [ ]:
print("Year-end portfolio weights:\n")
for m in METHODS:
    if m not in wt_hist: continue
    print(f"{LABELS[m]}:")
    disp = wt_hist[m].copy()
    for t in TICKERS: disp[t] = disp[t].map("{:.2%}".format)
    display(disp)
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for idx, m in enumerate(METHODS):
    if m not in wt_hist: continue
    ax = axes[idx]
    w  = wt_hist[m]
    x  = np.arange(len(w.index))
    bottoms = np.zeros(len(x))
    for t in TICKERS:
        ax.bar(x, w[t].values, bottom=bottoms, label=t,
               color=ASSET_COLORS[t], edgecolor="white", linewidth=0.5)
        bottoms += w[t].values
    ax.set_title(LABELS[m], fontsize=10, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(w.index.astype(str), fontsize=8)
    ax.set_ylabel("Weight"); ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.3)
    if idx == 0: ax.legend(fontsize=8, loc="upper right")

fig.suptitle("Portfolio Weights Through Time (Annual Rebalance)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## 9. Risk Contribution Analysis

**Key insight:** capital weight and risk contribution are not the same. A large weight in a low-volatility asset contributes less risk than its weight implies.

In [ ]:
# Use the 2019-2022 training covariance (matches Version 1 final year)
ret_2019_2022 = returns.loc["2019-01-01":"2022-12-28", TICKERS]
cov_final, alpha_final = ledoit_wolf_cov(ret_2019_2022)
print(f"Covariance estimated on 2019-2022 training data. LW shrinkage: {alpha_final:.4f}\n")

rc_data = {}
for m in METHODS:
    if m not in wt_hist: continue
    final_w = wt_hist[m].loc[2023].values
    rc = risk_contributions(final_w, cov_final)
    rc_data[m] = rc
    print(f"{LABELS[m]}:")
    for t, w, r in zip(TICKERS, final_w, rc):
        diff = r - w
        print(f"  {t:<6}  Weight: {w:6.2%}   Risk Contribution: {r:6.2%}   Diff: {diff:+.2%}")
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for idx, m in enumerate(METHODS):
    if m not in wt_hist or m not in rc_data: continue
    ax = axes[idx]
    final_w = wt_hist[m].loc[2023].values
    rc = rc_data[m]
    x  = np.arange(len(TICKERS))
    ax.bar(x - 0.2, final_w * 100, 0.35, label="Capital Weight (%)",
           color=[ASSET_COLORS[t] for t in TICKERS], alpha=0.9)
    ax.bar(x + 0.2, rc * 100, 0.35, label="Risk Contribution (%)",
           color=[ASSET_COLORS[t] for t in TICKERS], alpha=0.4,
           edgecolor=[ASSET_COLORS[t] for t in TICKERS], linewidth=1.5)
    ax.set_title(LABELS[m], fontsize=10, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(TICKERS)
    ax.set_ylabel("%"); ax.set_ylim(0, 65)
    ax.axhline(25, color="grey", linewidth=0.6, linestyle=":")
    ax.grid(axis="y", alpha=0.3)
    if idx == 0: ax.legend(fontsize=8)

fig.suptitle("Capital Weight vs Risk Contribution (2023 Final Weights)", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## 10. Performance Visualisations

In [ ]:
# Cumulative returns
fig, ax = plt.subplots(figsize=(14, 6))
for m in METHODS:
    if m not in port_rets: continue
    cum = (1 + port_rets[m]).cumprod()
    ax.plot(cum.index, cum, label=LABELS[m], color=COLORS[m], linewidth=2.0, linestyle=LS[m])

for i, yr in enumerate(TEST_YEARS):
    mask = port_rets[METHODS[0]].index.year == yr
    idx  = port_rets[METHODS[0]][mask].index
    if len(idx):
        ax.axvspan(idx[0], idx[-1], alpha=0.04, color="grey")
        ax.text(idx[len(idx)//2], ax.get_ylim()[0] * 1.02, str(yr),
                ha="center", fontsize=7.5, color="grey", alpha=0.8)

ax.axhline(1.0, color="black", linewidth=0.5, linestyle=":")
ax.set_title("Rolling Out-of-Sample Cumulative Returns (2019-2023)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Portfolio Value (Start = 1.0)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.savefig("outputs/figures/01_cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Drawdowns
fig, ax = plt.subplots(figsize=(14, 5))
for m in METHODS:
    if m not in port_rets: continue
    cum = (1 + port_rets[m]).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax()
    ax.fill_between(dd.index, dd * 100, 0, alpha=0.12, color=COLORS[m])
    ax.plot(dd.index, dd * 100, label=LABELS[m], color=COLORS[m], linewidth=1.5, linestyle=LS[m])

ax.set_title("Rolling Out-of-Sample Drawdowns (2019-2023)", fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Drawdown (%)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.savefig("outputs/figures/02_drawdowns.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Annual Sharpe bar chart
fig, ax = plt.subplots(figsize=(12, 5))
n = len(METHODS); x = np.arange(len(TEST_YEARS)); w = 0.18
for i, m in enumerate(METHODS):
    if m not in met_hist: continue
    vals   = met_hist[m]["Sharpe"].reindex(TEST_YEARS).fillna(0)
    offset = (i - n/2 + 0.5) * w
    ax.bar(x + offset, vals.values, w, label=LABELS[m], color=COLORS[m], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1.0, color="grey", linewidth=0.6, linestyle="--", alpha=0.5)
ax.set_title("Annual Sharpe Ratio by Strategy (2019-2023)", fontsize=12, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in TEST_YEARS])
ax.set_ylabel("Sharpe Ratio"); ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.savefig("outputs/figures/03_annual_sharpe.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Annual returns bar chart
fig, ax = plt.subplots(figsize=(12, 5))
for i, m in enumerate(METHODS):
    if m not in met_hist: continue
    vals   = met_hist[m]["Ann. Return"].reindex(TEST_YEARS).fillna(0)
    offset = (i - n/2 + 0.5) * w
    ax.bar(x + offset, vals.values * 100, w, label=LABELS[m], color=COLORS[m], alpha=0.85, edgecolor="white")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Annual Return by Strategy (2019-2023)", fontsize=12, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels([str(y) for y in TEST_YEARS])
ax.set_ylabel("Annual Return (%)"); ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.25)
plt.tight_layout(); plt.savefig("outputs/figures/04_annual_returns.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Turnover chart
if not turnover_df.empty:
    t_methods = [m for m in METHODS if m in turnover_df.columns]
    fig, ax = plt.subplots(figsize=(12, 4))
    x_to = np.arange(len(turnover_df.index))
    for i, m in enumerate(t_methods):
        offset = (i - len(t_methods)/2 + 0.5) * w
        ax.bar(x_to + offset, turnover_df[m].values * 100, w,
               label=LABELS[m], color=COLORS[m], alpha=0.85, edgecolor="white")
    ax.set_title("Portfolio Turnover by Year (One-Way %, includes year-boundary cost)", fontsize=11, fontweight="bold")
    ax.set_xticks(x_to); ax.set_xticklabels([str(y) for y in turnover_df.index])
    ax.set_ylabel("Turnover (%)"); ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.25)
    plt.tight_layout(); plt.savefig("outputs/figures/05_turnover.png", dpi=150, bbox_inches="tight")
    plt.show()

## 11. Transaction Cost Sensitivity

In [ ]:
tc_scenarios = [0, 5, 10, 25]
tc_results   = {}
print("Running transaction cost sensitivity...")
for tc in tc_scenarios:
    _, _, met_tc, _ = rolling_backtest(
        returns=returns, tickers=TICKERS, train_years=TRAIN_YEARS,
        test_years=TEST_YEARS, methods=METHODS, max_weight=MAX_WEIGHT,
        rebalance="annual", transaction_cost_bps=tc, verbose=False
    )
    tc_results[tc] = {m: met_tc[m]["Sharpe"].mean() for m in METHODS if m in met_tc}
    print(f"  {tc:2d} bps  done")

tc_df = pd.DataFrame(tc_results).T
tc_df.index.name = "TC (bps)"
tc_df.columns = [LABELS[m] for m in tc_df.columns]
print("\nAverage Annual Sharpe vs Transaction Cost:")
display(tc_df.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for m in METHODS:
    col = LABELS[m]
    if col in tc_df.columns:
        ax.plot(tc_df.index, tc_df[col], label=col, color=COLORS[m], linewidth=2,
                marker="o", markersize=5, linestyle=LS[m])
ax.set_title("Average Sharpe vs Transaction Cost Assumption", fontsize=12, fontweight="bold")
ax.set_xlabel("Transaction Cost (bps)"); ax.set_ylabel("Average Annual Sharpe (2019-2023)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
plt.tight_layout(); plt.savefig("outputs/figures/06_tc_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Rebalancing Frequency Sensitivity (Min Variance)

In [ ]:
rb_scenarios = ["none", "annual", "quarterly", "monthly"]
rb_results   = {}
print("Running rebalancing sensitivity (Min Variance)...")
for rb in rb_scenarios:
    _, _, met_rb, to_rb = rolling_backtest(
        returns=returns, tickers=TICKERS, train_years=TRAIN_YEARS,
        test_years=TEST_YEARS, methods=["min_variance"], max_weight=MAX_WEIGHT,
        rebalance=rb, transaction_cost_bps=TC_BPS, verbose=False
    )
    if "min_variance" in met_rb:
        m = met_rb["min_variance"]
        rb_results[rb] = {
            "Avg Sharpe":     round(m["Sharpe"].mean(), 4),
            "Avg Return":     f"{m['Ann. Return'].mean():.2%}",
            "Avg Volatility": f"{m['Ann. Volatility'].mean():.2%}",
            "Avg Max DD":     f"{m['Max Drawdown'].mean():.2%}",
            "Avg Turnover":   f"{to_rb['min_variance'].mean():.2%}" if "min_variance" in to_rb.columns else "n/a",
        }
    print(f"  {rb:<12} done")

rb_df = pd.DataFrame(rb_results).T
rb_df.index.name = "Rebalance"
print("\nMin Variance: Rebalancing Frequency Impact")
display(rb_df)

## 13. Interpretation

### Training window

At the start of each test year, the covariance matrix is estimated on a 4-year rolling window. For intra-year rebalancing tests, the window expands to include all data available up to the prior rebalance date. No future data is used.

### Transaction costs

Transaction costs are charged both at year-boundary rebalances (moving from prior year's final weights to new year's initial weights) and at any intra-year rebalance events. At 5 bps per unit of one-way turnover under annual rebalancing, costs are small but non-zero.

### Constraint asymmetry

The 40% per-asset cap is applied to Minimum Variance only, matching the Version 1 submitted specification. Risk Parity and Maximum Sharpe are shown unconstrained. BSV may exceed 50% in Risk Parity — this is mathematically correct because low-volatility assets need large capital weights to reach equal risk contribution.

### What the results show

**Minimum Variance** generally provides the most defensive profile, with lower volatility and shallower drawdowns in several periods. In strong equity years it tends to lag on return.

**Risk Parity** provides a balanced risk allocation and is a useful comparison to Minimum Variance, but its relative performance depends on the market regime.

**Maximum Sharpe** is input-sensitive. The concentrated allocations that emerge are a direct consequence of the historical expected return inputs, not a methodological error. This confirms the Version 1 finding: the return forecast is the main driver of instability, not the optimizer.

**Equal Weight** often carries higher equity-driven risk but can outperform in broad risk-on years.

No single strategy dominates every year. The ranking table in Section 7 provides the full-period comparison.

### Limitations

- Five out-of-sample years is more robust than one, but still insufficient for statistical significance testing across market regimes.
- Risk-free rate held at 0% throughout; in reality rates varied from near-zero (2019-2021) to above 5% (2023).
- No leverage, no shorting, no derivatives.
- Transaction costs are simplified; real costs also include bid-ask spread and market impact, especially for larger positions.

In [ ]:
# Final summary
print("=" * 72)
print("FULL-PERIOD SUMMARY (2019-2023, annual rebalancing, 5 bps TC)")
print("=" * 72)
display(agg_df)
print()
print("STRATEGY RANKING (Rank 1 = best per metric)")
display(rank_df)

## 14. Save Outputs

In [ ]:
import os
os.makedirs("outputs/tables",  exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

# Save performance tables
agg_raw.to_csv("outputs/tables/full_period_metrics.csv")
sharpe_tbl.to_csv("outputs/tables/annual_sharpe.csv")
ret_tbl.to_csv("outputs/tables/annual_returns.csv")
dd_tbl.to_csv("outputs/tables/annual_drawdown.csv")
rank_df.to_csv("outputs/tables/strategy_ranking.csv")
if not turnover_df.empty:
    turnover_df.to_csv("outputs/tables/turnover.csv")

print("Tables saved to outputs/tables/")
print("Charts already saved to outputs/figures/")

## Disclaimer

This notebook is for academic and educational purposes only. It does not constitute investment advice or a recommendation to buy or sell any security.